In [1]:
# 1. Imports, historical seasons, and optional persistent storage
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
import hashlib
import json
import re
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import requests
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

PROJECT_TITLE = "Against All Odds"
USE_GOOGLE_DRIVE = False  # Switch to True to store outputs in your Google Drive.
SOURCE_BASE = "https://www.football-data.co.uk/mmz4281"
SEASONS = {
    "2021-22": {"code": "2122", "split": "TRAIN"},
    "2022-23": {"code": "2223", "split": "TRAIN"},
    "2023-24": {"code": "2324", "split": "TRAIN"},
    "2024-25": {"code": "2425", "split": "VALIDATION"},
    "2025-26": {"code": "2526", "split": "TEST"},
}

if USE_GOOGLE_DRIVE and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/againstallodds")
elif Path.cwd().name.lower() in {"againstallodds", "against-all-odds", "against_all_odds"}:
    PROJECT_ROOT = Path.cwd().resolve()
elif "google.colab" in sys.modules:
    PROJECT_ROOT = Path("/content/againstallodds")
else:
    PROJECT_ROOT = Path.cwd().resolve() / "againstallodds"

RAW_DIR = PROJECT_ROOT / "data/raw/soccer"
CLEAN_DIR = PROJECT_ROOT / "data/clean/soccer"
DOCS_DIR = PROJECT_ROOT / "docs"
RESULTS_DIR = PROJECT_ROOT / "results"
for folder in (RAW_DIR, CLEAN_DIR, DOCS_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Planned EPL seasons:", ", ".join(SEASONS))

# 2. Collection helper: retry downloads and preserve original source bytes
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "AgainstAllOdds-StatisticsCapstone/1.0"})
source_manifest = []

def obtain_season_file(label, code):
    url = f"{SOURCE_BASE}/{code}/E0.csv"
    destination = RAW_DIR / f"premier_league_{code}_source.csv"
    if not destination.exists() or destination.stat().st_size == 0:
        last_error = None
        for attempt in range(1, 4):
            try:
                response = SESSION.get(url, timeout=45)
                response.raise_for_status()
                blob = response.content
                if len(blob) < 200 or blob.lstrip().lower().startswith(b"<html"):
                    raise ValueError("The URL returned HTML or an unexpectedly short file, not CSV data")
                pd.read_csv(BytesIO(blob), nrows=2)  # Do not cache an unparseable response.
                destination.write_bytes(blob)
                break
            except (requests.RequestException, ValueError, pd.errors.ParserError) as exc:
                last_error = exc
                if attempt < 3:
                    time.sleep(2 * attempt)
        else:
            raise RuntimeError(
                f"Could not obtain {label}: {url}. {last_error}. "
                f"For offline use, download E0.csv from that URL and save it as {destination.name} in {RAW_DIR}."
            )
    blob = destination.read_bytes()
    digest = hashlib.sha256(blob).hexdigest()
    try:
        frame = pd.read_csv(BytesIO(blob), low_memory=False)
    except Exception as exc:
        raise RuntimeError(f"Cached file for {label} is not readable: {destination}") from exc
    if frame.empty:
        raise RuntimeError(f"Source file has no matches: {destination}")
    source_manifest.append({
        "season": label, "season_code": code, "url": url,
        "file": str(destination.relative_to(PROJECT_ROOT)),
        "bytes": len(blob), "sha256": digest,
        "source_rows": len(frame), "source_columns": len(frame.columns),
        "retrieved_or_verified_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })
    print(f"{label}: {len(frame):,} raw rows, {len(frame.columns):,} columns")
    return frame

# 3. Collect all five seasons; inspect field coverage before dropping odds columns
raw_frames = []
for label, details in SEASONS.items():
    raw = obtain_season_file(label, details["code"])
    raw["season"] = label
    raw["season_code"] = details["code"]
    raw["data_split"] = details["split"]
    raw_frames.append(raw)

raw_all = pd.concat(raw_frames, ignore_index=True, sort=False)
manifest_df = pd.DataFrame(source_manifest)
display(manifest_df[["season", "source_rows", "source_columns", "url"]])

# Keep a separate source inventory; betting odds are not part of this project.
CORE_SOURCE = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
OPTIONAL_SOURCE = ["Time", "HS", "AS", "HST", "AST", "HF", "AF", "HC", "AC", "HY", "AY", "HR", "AR"]
field_audit = pd.DataFrame([
    {
        "season": label,
        "field": field,
        "priority": "required" if field in CORE_SOURCE else "optional",
        "available": field in frame.columns,
        "missing_pct": round(frame[field].isna().mean() * 100, 2) if field in frame.columns else np.nan,
    }
    for (label, _), frame in zip(SEASONS.items(), raw_frames)
    for field in CORE_SOURCE + OPTIONAL_SOURCE
])
if not field_audit.loc[field_audit["priority"].eq("required"), "available"].all():
    display(field_audit.loc[field_audit["priority"].eq("required") & ~field_audit["available"]])
    raise ValueError("A selected season lacks an essential match-result field; review the source before continuing.")
print("Required source fields are present in all five seasons.")
display(field_audit.pivot(index="field", columns="season", values="available"))

# 4. Normalize dates and clubs; preserve the source names for auditability
ALIASES = {
    "Man City": "Manchester City", "Man United": "Manchester United",
    "Newcastle": "Newcastle United", "Nott'm Forest": "Nottingham Forest",
    "Nottingham Forest": "Nottingham Forest", "Spurs": "Tottenham Hotspur",
    "Tottenham": "Tottenham Hotspur", "West Ham": "West Ham United",
    "Brighton": "Brighton & Hove Albion", "Leicester": "Leicester City",
    "Wolves": "Wolverhampton Wanderers", "Leeds": "Leeds United",
    "Sheffield United": "Sheffield United", "Sheffield Utd": "Sheffield United",
}

def parse_match_dates(series):
    # Explicit patterns prevent ambiguous dd/mm/yy dates from being read as mm/dd/yy.
    raw = series.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    for fmt in ("%d/%m/%Y", "%d/%m/%y", "%Y-%m-%d", "%d-%m-%Y", "%d-%m-%y"):
        mask = parsed.isna() & raw.notna()
        if mask.any():
            parsed.loc[mask] = pd.to_datetime(raw.loc[mask], format=fmt, errors="coerce")
    return parsed

keep_fields = CORE_SOURCE + OPTIONAL_SOURCE
matches = raw_all[[c for c in keep_fields if c in raw_all.columns] + ["season", "season_code", "data_split"]].copy()
matches["date_source"] = matches["Date"].astype("string")
matches["game_date"] = parse_match_dates(matches["Date"])
for side in ("Home", "Away"):
    raw_col = f"{side}Team"
    matches[f"{side.lower()}_team_source"] = matches[raw_col].astype("string").str.strip()
    matches[f"{side.lower()}_team"] = matches[f"{side.lower()}_team_source"].replace(ALIASES)

for col in ("FTHG", "FTAG") + tuple(c for c in OPTIONAL_SOURCE if c not in {"Time"}):
    if col in matches.columns:
        matches[col] = pd.to_numeric(matches[col], errors="coerce")
matches["FTR"] = matches["FTR"].astype("string").str.strip().str.upper()
print("Standardized team names and match dates.")
display(matches[["season", "game_date", "home_team", "away_team", "FTHG", "FTAG", "FTR"]].head(8))

# 5. Missing-value and duplicate audit (do not conceal failed matches)
critical = ["season", "season_code", "game_date", "home_team", "away_team", "FTHG", "FTAG", "FTR"]
missing_audit = pd.DataFrame({
    "field": matches.columns,
    "missing_count": matches.isna().sum().values,
    "missing_pct": (matches.isna().mean().values * 100).round(2),
}).sort_values(["missing_count", "field"], ascending=[False, True])

bad_critical = matches[critical].isna().any(axis=1)
invalid_score = (matches["FTHG"].lt(0) | matches["FTAG"].lt(0) |
                 matches["FTHG"].mod(1).ne(0) | matches["FTAG"].mod(1).ne(0))
bad_team = matches["home_team"].eq(matches["away_team"])

# A fixture is uniquely identified by season + ordered home/away teams (each pair meets once per venue).
fixture_key = ["season_code", "home_team", "away_team"]
duplicates = matches.duplicated(subset=fixture_key, keep="first")
issues = matches.loc[bad_critical | invalid_score | bad_team | duplicates,
                     ["season", "date_source", "home_team", "away_team", "FTHG", "FTAG", "FTR"]].copy()
issues["missing_critical"] = bad_critical.loc[issues.index].values
issues["invalid_score"] = invalid_score.loc[issues.index].values
issues["same_club"] = bad_team.loc[issues.index].values
issues["duplicate_fixture"] = duplicates.loc[issues.index].values

print("Raw rows:", len(matches), "| Duplicates:", int(duplicates.sum()),
      "| Critical/malformed rows:", int((bad_critical | invalid_score | bad_team).sum()))
display(missing_audit.head(25))
if not issues.empty:
    display(issues.head(20))

# Preserve every anomaly for review before excluding it.
valid_rows = ~(bad_critical | invalid_score | bad_team | duplicates)
clean = matches.loc[valid_rows].copy()
clean["FTHG"] = clean["FTHG"].astype(int)
clean["FTAG"] = clean["FTAG"].astype(int)

# 6. Verify recorded outcome against full-time scores; preserve draws
clean["computed_result"] = np.select(
    [clean["FTHG"].gt(clean["FTAG"]), clean["FTHG"].lt(clean["FTAG"])],
    ["H", "A"], default="D"
)
result_mismatch = clean["FTR"].ne(clean["computed_result"])
result_issue_rows = clean.loc[result_mismatch, ["season", "game_date", "home_team", "away_team", "FTHG", "FTAG", "FTR", "computed_result"]]
if len(result_issue_rows):
    print("⚠️ Source FTR conflicts with final goals. Review before modeling:")
    display(result_issue_rows)
    raise ValueError("Score/result inconsistencies found; fix source records before publishing the clean dataset.")

clean["result_category"] = clean["computed_result"].map(
    {"H": "HOME_WIN", "D": "DRAW", "A": "AWAY_WIN"}
)
clean["HOME_WIN"] = clean["computed_result"].eq("H").astype("int8")
clean["is_draw"] = clean["computed_result"].eq("D").astype("int8")
clean["home_is_home"] = 1
clean["away_is_home"] = 0
clean["home_goals"] = clean["FTHG"]
clean["away_goals"] = clean["FTAG"]
clean["goal_difference_home"] = clean["home_goals"] - clean["away_goals"]
clean["total_goals"] = clean["home_goals"] + clean["away_goals"]
clean["game_id"] = (
    "EPL_" + clean["season_code"] + "_" +
    clean["home_team"].str.upper().str.replace(r"[^A-Z0-9]", "", regex=True) + "_" +
    clean["away_team"].str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
)
clean = clean.sort_values(["game_date", "season_code", "game_id"]).reset_index(drop=True)
print("Home / Draw / Away counts:")
display(clean.groupby(["season", "result_category"]).size().unstack(fill_value=0))

# 7. Rename match-stat fields and build the final narrow game-level dataset
SOURCE_TO_CLEAN = {
    "HS": "home_shots", "AS": "away_shots",
    "HST": "home_shots_on_target", "AST": "away_shots_on_target",
    "HF": "home_fouls", "AF": "away_fouls",
    "HC": "home_corners", "AC": "away_corners",
    "HY": "home_yellow_cards", "AY": "away_yellow_cards",
    "HR": "home_red_cards", "AR": "away_red_cards",
}
clean = clean.rename(columns=SOURCE_TO_CLEAN)
identity_cols = [
    "game_id", "season", "season_code", "data_split", "game_date",
    "home_team", "away_team", "home_is_home", "away_is_home",
    "home_goals", "away_goals", "goal_difference_home", "total_goals",
    "result_category", "HOME_WIN", "is_draw",
]
optional_cols = [v for v in SOURCE_TO_CLEAN.values() if v in clean.columns]
# The final game-level CSV intentionally excludes betting odds and other unrelated fields.
epl_games = clean[identity_cols + optional_cols].copy()
assert epl_games["game_id"].is_unique, "Duplicated game_id: inspect naming rules."
print(f"Clean dataset: {len(epl_games):,} matches; {len(epl_games.columns)} columns")
display(epl_games.head(10))

# 8. Two rows per fixture: team-level post-match scoring and defensive statistics
home = epl_games[["game_id", "season", "data_split", "game_date", "home_team", "away_team", "home_goals", "away_goals", "result_category"]].rename(
    columns={"home_team": "team", "away_team": "opponent", "home_goals": "goals_scored", "away_goals": "goals_conceded"}
)
home["is_home"] = 1
home["team_result"] = home["result_category"].map({"HOME_WIN": "W", "DRAW": "D", "AWAY_WIN": "L"})
away = epl_games[["game_id", "season", "data_split", "game_date", "away_team", "home_team", "away_goals", "home_goals", "result_category"]].rename(
    columns={"away_team": "team", "home_team": "opponent", "away_goals": "goals_scored", "home_goals": "goals_conceded"}
)
away["is_home"] = 0
away["team_result"] = away["result_category"].map({"HOME_WIN": "L", "DRAW": "D", "AWAY_WIN": "W"})
for base, side in ((home, "home"), (away, "away")):
    for suffix, field in (("shots", "shots"), ("shots_on_target", "shots_on_target"),
                          ("fouls", "fouls"), ("corners", "corners"),
                          ("yellow_cards", "yellow_cards"), ("red_cards", "red_cards")):
        own = f"{side}_{field}"
        rival = f"{'away' if side == 'home' else 'home'}_{field}"
        if own in epl_games.columns:
            base[field] = epl_games[own].values
        if rival in epl_games.columns:
            base[f"{field}_allowed"] = epl_games[rival].values

team_games = pd.concat([home, away], ignore_index=True)
team_games["goal_difference"] = team_games["goals_scored"] - team_games["goals_conceded"]
team_games["points_earned"] = team_games["team_result"].map({"W": 3, "D": 1, "L": 0}).astype(int)
team_games = team_games.sort_values(["game_date", "game_id", "is_home"], ascending=[True, True, False]).reset_index(drop=True)
assert len(team_games) == len(epl_games) * 2
assert not team_games.duplicated(["game_id", "team"]).any()
print(f"Team-game dataset: {len(team_games):,} rows")
display(team_games.head(8))

# 9. Season/team coverage, match chronology, and final quality gates
season_rows = []
for season, part in epl_games.groupby("season", sort=True):
    participants = set(part["home_team"]).union(set(part["away_team"]))
    per_club = team_games.loc[team_games["season"].eq(season)].groupby("team")["game_id"].nunique()
    pair_counts = part.groupby(["home_team", "away_team"]).size()
    season_rows.append({
        "season": season, "matches": len(part), "expected_matches": 380,
        "unique_teams": len(participants), "expected_teams": 20,
        "every_team_38_matches": bool(len(per_club) == 20 and per_club.eq(38).all()),
        "unique_directed_fixtures": bool(pair_counts.eq(1).all()),
        "first_date": part["game_date"].min(), "last_date": part["game_date"].max(),
        "home_wins": int(part["result_category"].eq("HOME_WIN").sum()),
        "draws": int(part["is_draw"].sum()),
        "away_wins": int(part["result_category"].eq("AWAY_WIN").sum()),
    })
coverage = pd.DataFrame(season_rows)
coverage["complete_season"] = (
    coverage["matches"].eq(380) & coverage["unique_teams"].eq(20) &
    coverage["every_team_38_matches"] & coverage["unique_directed_fixtures"]
)
checks = {
    "five_completed_seasons": coverage["season"].nunique() == 5,
    "all_seasons_complete": coverage["complete_season"].all(),
    "no_critical_missing": epl_games[identity_cols].notna().all().all(),
    "unique_game_id": epl_games["game_id"].is_unique,
    "chronological_game_order": epl_games["game_date"].is_monotonic_increasing,
    "home_win_matches_score": epl_games["HOME_WIN"].eq(epl_games["home_goals"].gt(epl_games["away_goals"]).astype(int)).all(),
    "all_draws_labeled_zero": epl_games.loc[epl_games["is_draw"].eq(1), "HOME_WIN"].eq(0).all(),
    "team_game_rows_double": len(team_games) == len(epl_games) * 2,
    "no_source_result_disagreement": len(result_issue_rows) == 0,
    "no_removed_source_rows": len(issues) == 0,
}
quality_report = pd.DataFrame([{"check": k, "passed": bool(v)} for k, v in checks.items()])
display(coverage)
display(quality_report)
# Stop rather than calling an incomplete dataset 'clean' or writing an incomplete master CSV.
if not quality_report["passed"].all():
    raise RuntimeError("Week 5 quality checks failed; review source anomalies and coverage before saving clean data.")
print("✅ All five Premier League seasons and Week 5 structural checks passed")

# 10. Team-season table (descriptive only; end-of-season figures cannot predict earlier games)
team_season = team_games.groupby(["season", "data_split", "team"], as_index=False).agg(
    games=("game_id", "nunique"),
    wins=("team_result", lambda s: int(s.eq("W").sum())),
    draws=("team_result", lambda s: int(s.eq("D").sum())),
    losses=("team_result", lambda s: int(s.eq("L").sum())),
    league_points=("points_earned", "sum"),
    goals_for=("goals_scored", "sum"), goals_against=("goals_conceded", "sum"),
    goal_difference=("goal_difference", "sum"),
    average_goals_for=("goals_scored", "mean"), average_goals_against=("goals_conceded", "mean"),
)
team_season["win_pct"] = team_season["wins"] / team_season["games"]
team_season = team_season.sort_values(["season", "league_points", "goal_difference", "goals_for"], ascending=[True, False, False, False]).reset_index(drop=True)
print("Team-season summary: descriptive/post-season only")
display(team_season.head(20))

# 11. Extend Week 1 data dictionary without overwriting earlier weeks
DICT_COLUMNS = [
    "variable_name", "sport", "level", "category", "data_type", "description",
    "source", "pre_game_only", "calculation_or_definition", "missing_value_rule",
    "required_or_optional", "notes"
]
rows = []
def add_var(name, level, category, dtype, description, source, pre_game, calc, notes="", priority="required"):
    rows.append({
        "variable_name": name, "sport": "Premier League", "level": level,
        "category": category, "data_type": dtype, "description": description,
        "source": source, "pre_game_only": pre_game,
        "calculation_or_definition": calc,
        "missing_value_rule": "Required: no missing" if priority == "required" else "Optional: retain missing, audit by season",
        "required_or_optional": priority, "notes": notes,
    })

add_var("game_id", "match", "identifier", "string", "Unique match key", "Derived", "yes", "season_code + canonical home and away clubs")
add_var("game_date", "match", "time", "date", "Match calendar date", "Football-Data Date", "yes", "Parse source date using explicit day-first formats", "Kickoff time not guaranteed for every season")
add_var("season", "match", "time", "string", "Premier League season label", "Source URL", "yes", "YYYY-YY")
add_var("data_split", "match", "validation", "string", "Chronological dataset partition", "Week 2 season plan", "yes", "2021-24 train; 2024-25 validation; 2025-26 test")
for side in ("home", "away"):
    add_var(f"{side}_team", "match", "identifier", "string", f"Canonical {side} club", f"Football-Data {side.title()}Team", "yes", "Normalize documented name aliases")
    add_var(f"{side}_goals", "match", "outcome", "integer", f"Final {side} goals", "Football-Data FTHG/FTAG", "no", "Full-time score", "Post-match; NEVER a same-match input")
add_var("result_category", "match", "outcome", "string", "Three-way full-time result", "Derived", "no", "HOME_WIN / DRAW / AWAY_WIN", "Keep draws as a separate outcome")
add_var("HOME_WIN", "match", "outcome", "integer", "Binary target: home club won", "Derived", "no", "1 if home_goals > away_goals; else 0", "Draw and away win both equal 0")
add_var("is_draw", "match", "outcome", "integer", "Whether match finished level", "Derived", "no", "1 when home_goals == away_goals")
add_var("home_is_home", "match", "context", "integer", "Home-club indicator", "Derived", "yes", "Always 1")
add_var("away_is_home", "match", "context", "integer", "Away-club home indicator", "Derived", "yes", "Always 0")
add_var("goals_scored", "team_match", "offense", "integer", "Goals scored by one club in this match", "Football-Data final result", "no", "Take relevant home/away full-time score", "Prior match only for later rolling features")
add_var("goals_conceded", "team_match", "defense", "integer", "Goals allowed by one club in this match", "Football-Data final result", "no", "Take opponent full-time score", "Prior match only for later rolling features")
add_var("points_earned", "team_match", "outcome", "integer", "League points earned in match", "Derived", "no", "Win 3; draw 1; loss 0", "Prior match only for later rolling features")
for label in ("shots", "shots_on_target", "fouls", "corners", "yellow_cards", "red_cards"):
    add_var(label, "team_match", "match_stat", "numeric", f"Club {label.replace('_', ' ')} in the match", "Football-Data", "no", "Take observed side-specific match statistic", "Post-match, only shifted historical observations usable later", "optional")

new_dictionary = pd.DataFrame(rows, columns=DICT_COLUMNS)
previous = next((x for x in [DOCS_DIR / "data_dictionary_week4.csv", DOCS_DIR / "data_dictionary_week3.csv", DOCS_DIR / "data_dictionary_template.csv"] if x.exists()), None)
if previous is not None:
    prior_dictionary = pd.read_csv(previous, dtype="string").reindex(columns=DICT_COLUMNS)
    if set(["variable_name", "sport", "level"]).issubset(prior_dictionary.columns):
        keys = set(zip(new_dictionary.variable_name, new_dictionary.sport, new_dictionary.level))
        keep = [tuple(k) not in keys for k in prior_dictionary[["variable_name", "sport", "level"]].itertuples(index=False, name=None)]
        prior_dictionary = prior_dictionary.loc[keep]
    updated_dictionary = pd.concat([prior_dictionary, new_dictionary], ignore_index=True)
else:
    # This is an additive Week 5 dictionary; it never replaces earlier versions.
    updated_dictionary = new_dictionary
print(f"Week 5 dictionary variables: {len(new_dictionary)}; combined dictionary variables: {len(updated_dictionary)}")
display(new_dictionary.head(12))

# 12. Save Week 5 cleaned data and audit artifacts (only after all checks pass)
assert quality_report["passed"].all()
OUTPUTS = {
    "clean_game_csv": CLEAN_DIR / "premier_league_games_clean_2021_22_to_2025_26.csv",
    "clean_team_game_csv": CLEAN_DIR / "premier_league_team_games_clean_2021_22_to_2025_26.csv",
    "team_season_csv": CLEAN_DIR / "premier_league_team_season_summary_2021_22_to_2025_26.csv",
    "season_coverage_csv": RESULTS_DIR / "week5_epl_season_coverage.csv",
    "quality_report_csv": RESULTS_DIR / "week5_epl_quality_report.csv",
    "source_manifest_csv": DOCS_DIR / "week5_epl_source_manifest.csv",
    "source_field_audit_csv": DOCS_DIR / "week5_epl_field_audit.csv",
    "source_missingness_csv": DOCS_DIR / "week5_epl_missingness.csv",
    "source_issues_csv": DOCS_DIR / "week5_epl_source_issues.csv",
    "week5_dictionary_csv": DOCS_DIR / "data_dictionary_week5.csv",
}
for key, frame in (
    ("clean_game_csv", epl_games), ("clean_team_game_csv", team_games),
    ("team_season_csv", team_season), ("season_coverage_csv", coverage),
    ("quality_report_csv", quality_report), ("source_manifest_csv", manifest_df),
    ("source_field_audit_csv", field_audit), ("source_missingness_csv", missing_audit),
    ("source_issues_csv", issues), ("week5_dictionary_csv", updated_dictionary),
):
    frame.to_csv(OUTPUTS[key], index=False)
    print("Saved:", OUTPUTS[key].relative_to(PROJECT_ROOT))

# 13. Write source/cleaning notes, Week 5 README text, and reproduction instructions
summary_lines = []
for record in coverage.itertuples(index=False):
    summary_lines.append(
        f"- {record.season}: {record.matches} matches; {record.unique_teams} clubs; "
        f"home wins {record.home_wins}, draws {record.draws}, away wins {record.away_wins}."
    )

report = "\n".join([
    "# Against All Odds — Week 5 Premier League Data Collection and Cleaning",
    "",
    "Source: https://www.football-data.co.uk/englandm.php",
    "Per-season CSV: https://www.football-data.co.uk/mmz4281/{season_code}/E0.csv",
    "Historical window: 2021–22 through 2025–26 (five completed seasons).",
    "Chronological split: 2021–22 to 2023–24 TRAIN; 2024–25 VALIDATION; 2025–26 TEST.",
    "",
    "## Coverage checked in this run",
    *summary_lines,
    "",
    "## Processing",
    "- Preserve original source files and document SHA-256 checksums.",
    "- Normalize club aliases; parse calendar dates using explicit day-first formats.",
    "- Check missing fields, duplicate season/home/away fixtures, final result consistency, and 380 matches/20 clubs/38 fixtures per club.",
    "- Produce one row per match and two team-match rows per fixture; retain full-time H/D/A result and binary HOME_WIN.",
    "- Keep post-match goals, shots and discipline stats for future historical rolling features only.",
    "",
    "## Outcome definition and leakage safeguards",
    "HOME_WIN = 1 for home victory; HOME_WIN = 0 for draw or away victory. Retain result_category and is_draw.",
    "Never use this match's goals, result, shots, or season-end summary to predict this same match.",
    "When generating pre-match features later, sort within team chronologically and use only earlier completed matches (e.g., shift(1) before rolling), with kickoff timestamps where needed for same-day ordering.",
    "",
    "## Source limitations",
    "Some optional statistics differ by season; expected goals (xG), historical injury status, and complete kickoff timestamps are not supplied as guaranteed columns.",
    "Only the Premier League was collected. Optional European leagues remain outside Week 5 scope.",
    "Betting odds are excluded deliberately from the cleaned datasets.",
    "",
    "## Reproduction",
    "Open notebooks/06_soccer_analysis.ipynb in Google Colab and choose Runtime > Run all.",
    "If keeping files between sessions, enable USE_GOOGLE_DRIVE or download the ZIP produced by the final cell.",
])
notes_path = DOCS_DIR / "week5_premier_league_source_and_cleaning.md"
notes_path.write_text(report + "\n", encoding="utf-8")
readme_addition = "\n".join([
    "## Week 5 — Premier League data collection and cleaning",
    "Collected the 2021–22 through 2025–26 Premier League seasons from Football-Data.co.uk.",
    "Created clean match, team-match, and team-season datasets; standardized club names and dates; checked match coverage, duplicates, outcomes and missing values.",
    "For binary prediction, HOME_WIN is 1 for a home win and 0 for a draw or an away win; the three-way outcome remains available for analysis.",
    "Source, data dictionary, cleaning, and quality checks are documented in docs/week5_premier_league_source_and_cleaning.md.",
    "Current-match results and match statistics must never be used as same-match pre-game predictors.",
])
readme_path = DOCS_DIR / "week5_README_addition.md"
readme_path.write_text(readme_addition + "\n", encoding="utf-8")
print("Saved:", notes_path.relative_to(PROJECT_ROOT))
print("Saved:", readme_path.relative_to(PROJECT_ROOT))

# 14. Pack the Week 5 outputs for downloading and pushing to GitHub
ZIP_PATH = PROJECT_ROOT / "week5_epl_outputs.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((CLEAN_DIR.glob("*.csv"))):
        if "premier_league" in path.name:
            archive.write(path, path.relative_to(PROJECT_ROOT))
    for folder in (DOCS_DIR, RESULTS_DIR):
        for path in sorted(folder.glob("*week5*")):
            if path.is_file():
                archive.write(path, path.relative_to(PROJECT_ROOT))
print(f"✅ WEEK 5 COMPLETE: {len(epl_games):,} matches; {len(team_games):,} team-match records")
print(f"ZIP: {ZIP_PATH}")
print("Upload 06_soccer_analysis.ipynb to notebooks/; docs/week5* to docs/; results/week5* to results/.")
print("For public GitHub, consider uploading reproduction instructions rather than raw source/redistributed datasets.")
if "google.colab" in sys.modules and not USE_GOOGLE_DRIVE:
    print("Colab runtime files are temporary. In the Files sidebar, right-click week5_epl_outputs.zip > Download.")


Project root: /content/againstallodds
Planned EPL seasons: 2021-22, 2022-23, 2023-24, 2024-25, 2025-26
2021-22: 380 raw rows, 106 columns
2022-23: 380 raw rows, 106 columns
2023-24: 380 raw rows, 106 columns
2024-25: 380 raw rows, 120 columns
2025-26: 380 raw rows, 132 columns


,season,source_rows,source_columns,url
0,2021-22,380,106,https://www.football-data.co.uk/mmz4281/2122/E...
1,2022-23,380,106,https://www.football-data.co.uk/mmz4281/2223/E...
2,2023-24,380,106,https://www.football-data.co.uk/mmz4281/2324/E...
3,2024-25,380,120,https://www.football-data.co.uk/mmz4281/2425/E...
4,2025-26,380,132,https://www.football-data.co.uk/mmz4281/2526/E...


Required source fields are present in all five seasons.


season,2021-22,2022-23,2023-24,2024-25,2025-26
field,,,,,
AC,True,True,True,True,True
AF,True,True,True,True,True
AR,True,True,True,True,True
AS,True,True,True,True,True
AST,True,True,True,True,True
AY,True,True,True,True,True
AwayTeam,True,True,True,True,True
Date,True,True,True,True,True
FTAG,True,True,True,True,True


Standardized team names and match dates.


...

✅ All five Premier League seasons and Week 5 structural checks passed
Team-season summary: descriptive/post-season only


,season,data_split,team,games,wins,draws,losses,league_points,goals_for,goals_against,goal_difference,average_goals_for,average_goals_against,win_pct
0,2021-22,TRAIN,Manchester City,38,29,6,3,93,99,26,73,2.605263,0.684211,0.763158
1,2021-22,TRAIN,Liverpool,38,28,8,2,92,94,26,68,2.473684,0.684211,0.736842
2,2021-22,TRAIN,Chelsea,38,21,11,6,74,76,33,43,2.000000,0.868421,0.552632
3,2021-22,TRAIN,Tottenham Hotspur,38,22,5,11,71,69,40,29,1.815789,1.052632,0.578947
4,2021-22,TRAIN,Arsenal,38,22,3,13,69,61,48,13,1.605263,1.263158,0.578947
5,2021-22,TRAIN,Manchester United,38,16,10,12,58,57,57,0,1.500000,1.500000,0.421053
6,2021-22,TRAIN,West Ham United,38,16,8,14,56,60,51,9,1.578947,1.342105,0.421053
7,2021-22,TRAIN,Leicester City,38,14,10,14,52,62,59,3,1.631579,1.552632,0.368421
8,2021-22,TRAIN,Brighton & Hove Albion,38,12,15,11,51,42,44,-2,1.105263,1.157895,0.315789
9,2021-22,TRAIN,Wolverhampton Wanderers,38,15,6,17,51,38,43,-5,1.000000,1.131579,0.394737


Week 5 dictionary variables: 22; combined dictionary variables: 22


,variable_name,sport,level,category,data_type,description,source,pre_game_only,calculation_or_definition,missing_value_rule,required_or_optional,notes
0,game_id,Premier League,match,identifier,string,Unique match key,Derived,yes,season_code + canonical home and away clubs,Required: no missing,required,
1,game_date,Premier League,match,time,date,Match calendar date,Football-Data Date,yes,Parse source date using explicit day-first for...,Required: no missing,required,Kickoff time not guaranteed for every season
2,season,Premier League,match,time,string,Premier League season label,Source URL,yes,YYYY-YY,Required: no missing,required,
3,data_split,Premier League,match,validation,string,Chronological dataset partition,Week 2 season plan,yes,2021-24 train; 2024-25 validation; 2025-26 test,Required: no missing,required,
4,home_team,Premier League,match,identifier,string,Canonical home club,Football-Data HomeTeam,yes,Normalize documented name aliases,Required: no missing,required,
5,home_goals,Premier League,match,outcome,integer,Final home goals,Football-Data FTHG/FTAG,no,Full-time score,Required: no missing,required,Post-match; NEVER a same-match input
6,away_team,Premier League,match,identifier,string,Canonical away club,Football-Data AwayTeam,yes,Normalize documented name aliases,Required: no missing,required,
7,away_goals,Premier League,match,outcome,integer,Final away goals,Football-Data FTHG/FTAG,no,Full-time score,Required: no missing,required,Post-match; NEVER a same-match input
8,result_category,Premier League,match,outcome,string,Three-way full-time result,Derived,no,HOME_WIN / DRAW / AWAY_WIN,Required: no missing,required,Keep draws as a separate outcome
9,HOME_WIN,Premier League,match,outcome,integer,Binary target: home club won,Derived,no,1 if home_goals > away_goals; else 0,Required: no missing,required,Draw and away win both equal 0


Saved: data/clean/soccer/premier_league_games_clean_2021_22_to_2025_26.csv
Saved: data/clean/soccer/premier_league_team_games_clean_2021_22_to_2025_26.csv
Saved: data/clean/soccer/premier_league_team_season_summary_2021_22_to_2025_26.csv
Saved: results/week5_epl_season_coverage.csv
Saved: results/week5_epl_quality_report.csv
Saved: docs/week5_epl_source_manifest.csv
Saved: docs/week5_epl_field_audit.csv
Saved: docs/week5_epl_missingness.csv
Saved: docs/week5_epl_source_issues.csv
Saved: docs/data_dictionary_week5.csv
Saved: docs/week5_premier_league_source_and_cleaning.md
Saved: docs/week5_README_addition.md
✅ WEEK 5 COMPLETE: 1,900 matches; 3,800 team-match records
ZIP: /content/againstallodds/week5_epl_outputs.zip
Upload 06_soccer_analysis.ipynb to notebooks/; docs/week5* to docs/; results/week5* to results/.
For public GitHub, consider uploading reproduction instructions rather than raw source/redistributed datasets.
Colab runtime files are temporary. In the Files sidebar, right-cli